In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

In [2]:
import tensorflow as tf

model = tf.keras.models.load_model("../backend/model/best_food_model.keras")

print("✅ Model loaded successfully!")

✅ Model loaded successfully!


In [5]:
nutrition_df = pd.read_csv("../backend/dataset/nutrition_dataset.csv")

nutrition_df.head()

,Food,Category,Serving_Size,Calories_kcal,Protein_g,Carbohydrates_g,Fat_g,Fiber_g,Sugar_g,Sodium_mg,Meal_Type,Best_For,Health_Rating,Nutrition_Score,Glycemic_Index,Allergens,Is_Veg,Description
0,Aloo_matar,Main Course,1 bowl (200 g),215,6.8,28.5,8.9,7.2,4.8,420,Lunch/Dinner,Lunch,Healthy,88,Medium,No Allergens,Yes,North Indian curry made with potatoes and gree...
1,Besan_cheela,Breakfast,2 cheelas (150 g),295,13.5,26.0,15.2,6.1,3.2,380,Breakfast,Breakfast,Healthy,86,Low,No Allergens,Yes,Protein-rich gram flour pancake with Indian sp...
2,Biryani,Main Course,1 plate (300 g),470,16.2,56.8,19.8,3.4,2.8,890,Lunch/Dinner,Lunch,Moderate,71,High,Depends on Ingredients,Depends on preparation,"Fragrant rice dish cooked with vegetables,chic..."
3,Chapathi,Bread,2 chapathis (80 g),220,7.2,43.5,3.0,6.4,1.0,240,Breakfast/Lunch/Dinner,Any Meal,Healthy,92,Medium,Gluten,Yes,Whole wheat flatbread served with Indian curries.
4,Chole_bature,Main Course,1 plate (350 g),680,18.0,74.5,33.5,13.2,8.4,980,Lunch,Lunch,Indulgent,46,High,Gluten,Yes,Spiced chickpeas served with deep-fried bread.


In [4]:
food_classes = sorted([
    "Aloo_matar",
    "Besan_cheela",
    "Biryani",
    "Chapathi",
    "Chole_bature",
    "Dahl",
    "Dhokla",
    "Dosa",
    "Gulab_jamun",
    "Idli",
    "Jalebi",
    "Kadai_paneer",
    "Naan",
    "Paani_puri",
    "Pakoda",
    "Pav_bhaji",
    "Poha",
    "Rolls",
    "Samosa",
    "Vada_pav"
])

In [17]:
def predict_food_with_nutrition(image_path):

    image = Image.open(image_path).convert("RGB")
    image = image.resize((224, 224))

    image_array = np.array(image) / 255.0
    image_array = np.expand_dims(image_array, axis=0)

    prediction = model.predict(image_array, verbose=0)

    confidence = np.max(prediction)
    predicted_index = np.argmax(prediction)

    # Reject unknown foods
    if confidence < 0.45:
        return None, confidence, None

    food = food_classes[predicted_index]

    nutrition = nutrition_df[
        nutrition_df["Food"] == food
    ].iloc[0]

    return food, confidence, nutrition

In [20]:
from pathlib import Path
import random

food_name = "Biryani"      # Change to any class

folder = Path(r"..\dataset\raw\Indian_Food_Dataset") / food_name

image_path = str(random.choice(list(folder.glob("*"))))

print(image_path)

..\dataset\raw\Indian_Food_Dataset\Biryani\00000198_resized.png


In [21]:
def generate_recommendation(food_name, nutrition):

    recommendations = []

    # ===========================
    # Nutrition-based suggestions
    # ===========================

    if nutrition["Calories_kcal"] > 450:
        recommendations.append("⚠ High-calorie meal. Consider smaller portions if you're watching your calorie intake.")
    elif nutrition["Calories_kcal"] < 200:
        recommendations.append("✅ Light meal with relatively low calories.")

    if nutrition["Protein_g"] >= 15:
        recommendations.append("💪 Rich in protein, supporting muscle maintenance and recovery.")
    elif nutrition["Protein_g"] < 6:
        recommendations.append("🥛 Pair with a protein-rich food like curd, paneer, eggs, or dal.")

    if nutrition["Fiber_g"] >= 6:
        recommendations.append("🌾 High fiber helps improve digestion and keeps you full longer.")

    if nutrition["Sugar_g"] >= 20:
        recommendations.append("🍬 High sugar content. Best consumed occasionally.")

    if nutrition["Sodium_mg"] >= 700:
        recommendations.append("🧂 High sodium. People with high blood pressure should limit intake.")

    if nutrition["Health_Rating"] == "Healthy":
        recommendations.append("🥗 Excellent choice for regular consumption.")
    elif nutrition["Health_Rating"] == "Indulgent":
        recommendations.append("⚠ Enjoy occasionally as part of a balanced diet.")

    # ===========================
    # Food-specific suggestions
    # ===========================

    food_tips = {

        "Idli":
            "🍲 Pair with sambar for extra protein and fiber.",

        "Dosa":
            "🥥 Serve with coconut chutney and sambar for a balanced meal.",

        "Poha":
            "🥜 Adding peanuts increases protein and healthy fats.",

        "Besan_cheela":
            "🥬 Add vegetables like spinach or capsicum for more nutrients.",

        "Dahl":
            "🍚 Combine with brown rice or chapathi for a complete meal.",

        "Chapathi":
            "🥗 Pair with dal or vegetable curry for balanced nutrition.",

        "Biryani":
            "🥗 Serve with raita and fresh salad to improve the nutritional balance.",

        "Kadai_paneer":
            "🥦 Pair with whole wheat chapathi instead of naan for a healthier meal.",

        "Pav_bhaji":
            "🧈 Ask for less butter and include extra vegetables if possible.",

        "Rolls":
            "🥗 Add fresh vegetables and avoid excess sauces.",

        "Paani_puri":
            "💧 Ensure hygienic preparation and enjoy in moderation due to high sodium.",

        "Samosa":
            "🥗 Pair with salad instead of sugary beverages to balance the meal.",

        "Pakoda":
            "☕ Enjoy occasionally with tea; avoid frequent deep-fried snacks.",

        "Vada_pav":
            "🥗 Pair with fresh salad to improve overall nutrition.",

        "Chole_bature":
            "⚠ Very filling meal. Limit portion size and avoid frequent consumption.",

        "Naan":
            "🌾 Whole wheat chapathi is a healthier alternative for regular meals.",

        "Jalebi":
            "🍬 Best enjoyed occasionally after a balanced meal due to its high sugar content.",

        "Gulab_jamun":
            "🍨 Consume as an occasional dessert because of its high sugar and calorie content.",

        "Dhokla":
            "🥗 Healthy steamed snack that pairs well with green chutney.",

        "Aloo_matar":
            "🥗 Pair with whole wheat chapathi instead of rice for better blood sugar control."
    }

    if food_name in food_tips:
        recommendations.append(food_tips[food_name])

    # ===========================
    # Best meal timing
    # ===========================

    recommendations.append(
        f"⏰ Best consumed during: {nutrition['Best_For']}."
    )

    return recommendations

In [24]:
food, confidence, nutrition = predict_food_with_nutrition(image_path)

if food is None:

    print("="*65)
    print("⚠ FOOD NOT SUPPORTED")
    print("="*65)

    print(f"Model Confidence : {confidence*100:.2f}%")

    print("\nSorry! NutritionAI currently supports only these 20 Indian dishes:\n")

    for item in sorted(food_classes):
        print("•", item)

else:

    recommendations = generate_recommendation(food, nutrition)

    print("="*65)
    print("🍽 NUTRITION AI RESULT")
    print("="*65)

    print(f"Predicted Food : {food}")
    print(f"Confidence     : {confidence*100:.2f}%")

    print("\n📊 NUTRITION FACTS")
    print("-"*65)

    print(f"Serving Size     : {nutrition['Serving_Size']}")
    print(f"Calories         : {nutrition['Calories_kcal']} kcal")
    print(f"Protein          : {nutrition['Protein_g']} g")
    print(f"Carbohydrates    : {nutrition['Carbohydrates_g']} g")
    print(f"Fat              : {nutrition['Fat_g']} g")
    print(f"Fiber            : {nutrition['Fiber_g']} g")
    print(f"Sugar            : {nutrition['Sugar_g']} g")
    print(f"Sodium           : {nutrition['Sodium_mg']} mg")

    print("\n❤️ HEALTH")
    print("-"*65)

    print(f"Nutrition Score  : {nutrition['Nutrition_Score']}/100")
    print(f"Health Rating    : {nutrition['Health_Rating']}")
    print(f"Glycemic Index   : {nutrition['Glycemic_Index']}")
    print(f"Allergens        : {nutrition['Allergens']}")
    print(f"Vegetarian       : {nutrition['Is_Veg']}")

    print("\n🤖 AI RECOMMENDATION")
    print("-"*65)

    for tip in recommendations:
        print(tip)

    print("\n📝 Description")
    print("-"*65)
    print(nutrition["Description"])

🍽 NUTRITION AI RESULT
Predicted Food : Biryani
Confidence     : 99.99%

📊 NUTRITION FACTS
-----------------------------------------------------------------
Serving Size     : 1 plate (300 g)
Calories         : 470 kcal
Protein          : 16.2 g
Carbohydrates    : 56.8 g
Fat              : 19.8 g
Fiber            : 3.4 g
Sugar            : 2.8 g
Sodium           : 890 mg

❤️ HEALTH
-----------------------------------------------------------------
Nutrition Score  : 71/100
Health Rating    : Moderate
Glycemic Index   : High
Allergens        : Depends on Ingredients
Vegetarian       : Depends on preparation

🤖 AI RECOMMENDATION
-----------------------------------------------------------------
⚠ High-calorie meal. Consider smaller portions if you're watching your calorie intake.
💪 Rich in protein, supporting muscle maintenance and recovery.
🧂 High sodium. People with high blood pressure should limit intake.
🥗 Serve with raita and fresh salad to improve the nutritional balance.
⏰ Best consu